# Ablation Condition 1: 3-Agent Sequential (No RAG/KG)

Three separate LLM calls, one per subsystem, with no retrieval-augmented generation or knowledge graph access.
Each agent produces the output of one MARS subsystem using only parametric LLM knowledge.

- **Agent 1**: Property Extraction (produces properties + constraints)
- **Agent 2**: Material Discovery (proposes a candidate material)
- **Agent 3**: Manufacturability Assessment (produces process recipe or blocking constraints)

Output of each agent is fed as input to the next, preserving the sequential data flow of the full MARS pipeline.

## Setup

In [1]:
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime

current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "config").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "config").exists():
    project_root = current_dir.parent
else:
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root: {project_root}")

from src.config import load_config, load_prompts
from src.utils import (
    llm,
    load_ablation_queries,
    extract_json_from_response,
    build_ablation_evaluation,
    save_ablation_result,
)

config = load_config()
prompts = load_prompts()
print("Configuration and prompts loaded")

project_root: /orcd/pool/004/tphage/SG_march/MARS_ablation2
Configuration and prompts loaded


## Initialize LLM

In [2]:
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"],
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
temperature = config["llm"].get("temperature", 0)
print(f"LLM initialized: {config['llm']['model_name']}")

LLM initialized: gpt-oss-20b


## Load Queries and Prompts

In [3]:
queries = load_ablation_queries()
ablation_prompts = prompts["ablation"]

print(f"Loaded {len(queries)} benchmark queries:")
for q in queries:
    print(f"  - {q['name']}: {q['sentence'][:80]}...")

Loaded 6 benchmark queries:
  - Query1: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetr...
  - Query2: Identify PFAS free thermoplastic material for heat shrink tubing with a 2:1 shri...
  - Query3: Identify non fluoropolymer thermoplastics that can withstand severe thermal shoc...
  - Query4: What non-PFAS additives or formulation strategies can be used in thermoplastics ...
  - Query5: Beyond PEEK, which thermoplastic materials can:
- Withstand continuous or interm...
  - PTFE_Seals: Find a substitute material for PTFE that can be used in industrial seals applica...


## Define 3-Agent Pipeline

In [4]:
def run_3agent_ablation(query, generate_fn, ablation_prompts, temperature=0):
    """Run the 3-agent sequential ablation for a single query."""
    raw_responses = {}
    start_time = time.time()

    # ---- Agent 1: Property Extraction ----
    print("  Agent 1: Property Extraction...")
    a1_system = ablation_prompts["agent1_properties"]
    a1_user = ablation_prompts["agent1_properties_user_prompt"].format(
        sentence=query["sentence"],
        material_X=query["material_X"],
        application_Y=query["application_Y"],
    )
    a1_raw = generate_fn(system_prompt=a1_system, prompt=a1_user, temperature=temperature)
    raw_responses["agent1_properties"] = a1_raw
    a1_parsed = extract_json_from_response(a1_raw)

    if a1_parsed is None:
        print("    WARNING: Failed to parse Agent 1 response as JSON")
        a1_parsed = {"properties": [], "constraints": []}

    properties = a1_parsed.get("properties", [])
    constraints = a1_parsed.get("constraints", [])
    print(f"    Extracted {len(properties)} properties, {len(constraints)} constraints")

    # ---- Agent 2: Material Discovery ----
    print("  Agent 2: Material Discovery...")
    a2_system = ablation_prompts["agent2_candidate"]
    a2_user = ablation_prompts["agent2_candidate_user_prompt"].format(
        material_X=query["material_X"],
        application_Y=query["application_Y"],
        properties_json=json.dumps(properties, indent=2),
        constraints_json=json.dumps(constraints, indent=2),
    )
    a2_raw = generate_fn(system_prompt=a2_system, prompt=a2_user, temperature=temperature)
    raw_responses["agent2_candidate"] = a2_raw
    a2_parsed = extract_json_from_response(a2_raw)

    if a2_parsed is None:
        print("    WARNING: Failed to parse Agent 2 response as JSON")
        a2_parsed = {"material_name": "UNKNOWN", "material_class": "unknown", "justification": a2_raw}

    candidate = a2_parsed
    print(f"    Proposed candidate: {candidate.get('material_name', 'UNKNOWN')}")

    # ---- Agent 3: Manufacturability Assessment ----
    print("  Agent 3: Manufacturability Assessment...")
    a3_system = ablation_prompts["agent3_manufacturing"]
    a3_user = ablation_prompts["agent3_manufacturing_user_prompt"].format(
        material_name=candidate.get("material_name", "UNKNOWN"),
        material_class=candidate.get("material_class", "unknown"),
        application_Y=query["application_Y"],
        justification=candidate.get("justification", ""),
        properties_json=json.dumps(properties, indent=2),
        constraints_json=json.dumps(constraints, indent=2),
    )
    a3_raw = generate_fn(system_prompt=a3_system, prompt=a3_user, temperature=temperature)
    raw_responses["agent3_manufacturing"] = a3_raw
    a3_parsed = extract_json_from_response(a3_raw)

    if a3_parsed is None:
        print("    WARNING: Failed to parse Agent 3 response as JSON")
        a3_parsed = {
            "status": "unknown",
            "process_recipe": None,
            "blocking_constraints": [],
            "feedback_to_system2": a3_raw,
        }

    manufacturing = a3_parsed
    print(f"    Manufacturing status: {manufacturing.get('status', 'unknown')}")

    duration = time.time() - start_time

    return build_ablation_evaluation(
        query=query,
        properties=properties,
        constraints=constraints,
        candidate=candidate,
        manufacturing=manufacturing,
        condition_name="3agent",
        run_id=datetime.now().strftime("%Y%m%d%H"),
        duration_seconds=duration,
        raw_responses=raw_responses,
    )

## Run All Queries

In [5]:
results = {}
output_base = os.path.join(project_root, "pipeline_logs")

for i, query in enumerate(queries, 1):
    name = query["name"]
    print(f"\n{'='*70}")
    print(f"[{i}/{len(queries)}] Running 3-Agent ablation for: {name}")
    print(f"{'='*70}")
    print(f"  Query: {query['sentence'][:100]}...")

    result = run_3agent_ablation(query, generate, ablation_prompts, temperature=temperature)
    results[name] = result

    output_dir = os.path.join(output_base + f"_{name}")
    path = save_ablation_result(result, output_dir, "3agent", result["metadata"]["pipeline_run_id"])
    print(f"  Saved to: {path}")
    print(f"  Duration: {result['metadata']['duration_seconds']:.1f}s")
    print(f"  Candidate: {result['candidate_selection']['final_candidate'].get('material_name', 'N/A') if result['candidate_selection']['final_candidate'] else 'N/A'}")
    print(f"  Mfg status: {result['manufacturing_process']['status']}")

print(f"\n{'='*70}")
print("All 3-Agent ablation runs complete.")
print(f"{'='*70}")


[1/6] Running 3-Agent ablation for: Query1
  Query: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetrafluoroethylene, hex...
  Agent 1: Property Extraction...
    Extracted 11 properties, 6 constraints
  Agent 2: Material Discovery...
    Proposed candidate: Arkema FPU‑1000 (Fluorinated Polyurethane Elastomer)
  Agent 3: Manufacturability Assessment...
    Manufacturing status: manufacturable
  Saved to: /orcd/pool/004/tphage/SG_march/MARS_ablation2/pipeline_logs_Query1/ablation_3agent_2026032415.json
  Duration: 227.2s
  Candidate: Arkema FPU‑1000 (Fluorinated Polyurethane Elastomer)
  Mfg status: manufacturable

[2/6] Running 3-Agent ablation for: Query2
  Query: Identify PFAS free thermoplastic material for heat shrink tubing with a 2:1 shrink ratio that could ...
  Agent 1: Property Extraction...
    Extracted 8 properties, 6 constraints
  Agent 2: Material Discovery...
    Proposed candidate: Kurt J. Schaefer PEKK‑HT Heat Shrink Tubing
  Agent 3: Ma

## Summary

In [6]:
print(f"{'Query':<20} {'Candidate':<50} {'Status':<15} {'Time (s)':<10}")
print("-" * 95)
for name, result in results.items():
    cand = result["candidate_selection"]["final_candidate"]
    cand_name = (cand.get("material_name", "N/A") if cand else "N/A")[:48]
    status = result["manufacturing_process"]["status"]
    dur = result["metadata"]["duration_seconds"]
    print(f"{name:<20} {cand_name:<50} {status:<15} {dur:<10.1f}")

Query                Candidate                                          Status          Time (s)  
-----------------------------------------------------------------------------------------------
Query1               Arkema FPU‑1000 (Fluorinated Polyurethane Elasto   manufacturable  227.2     
Query2               Kurt J. Schaefer PEKK‑HT Heat Shrink Tubing        manufacturable  91.9      
Query3               Vespel SP‑22 (PEEK)                                manufacturable  74.8      
Query4               Polydimethylsiloxane Powder (Dow Corning 2000)     manufacturable  71.4      
Query5               Ultem 5000 (Polyetherimide)                        manufacturable  92.1      
PTFE_Seals           DuPont™ PEEK‑HT (Polyether Ether Ketone, high te   manufacturable  46.3      
